# Étape 1 — Diagnostic du PSO CUDA

Ce notebook :
1. vérifie l'environnement GPU (Colab, GPU T4)
2. clone/synchronise le dépôt `DE_CUDA`
3. compile et exécute le PSO fourni par le prof
4. pose 5 questions de compréhension (section 1.6) à répondre avant de passer à l'étape 2 (DE séquentiel)

**Avant d'exécuter** : Exécution > Modifier le type d'exécution > GPU (T4).

## 1. Vérifier le GPU

In [ ]:
!nvidia-smi

In [ ]:
!nvcc --version

## 2. Cloner / mettre à jour le dépôt

Nécessite le secret `GITHUB_TOKEN` configuré dans les Secrets Colab (tâche 0.3).

In [ ]:
from google.colab import userdata
import os

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
GITHUB_USER = "Lounismsr"
REPO = "DE_CUDA"

repo_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO}.git"

%cd /content
if not os.path.exists(REPO):
    !git clone {repo_url}
%cd {REPO}
!git config user.name "Ton Nom"
!git config user.email "ton.email@uha.fr"
!git pull

## 3. Compilation

In [ ]:
!nvcc -o pso src/main.cpp src/kernel.cpp src/kernel.cu

## 4. Exécution

Livrable de la tâche 1.1 : copie la sortie de cette cellule (temps GPU + minimum trouvé).

In [ ]:
!./pso

## 1.3 — Répartition du temps (résultat, GPU Tesla T4, 30000 itérations)

| Partie | Temps | % |
|---|---|---|
| `kernelUpdateParticle` | 238.69 ms | 1.81 % |
| `kernelUpdatePBest` | 2617.56 ms | 19.86 % |
| `cudaMemcpy` pBest D2H | 380.54 ms | 2.89 % |
| **Boucle CPU gBest** | **8572.33 ms** | **65.05 %** |
| `cudaMemcpy` gBest H2D | 308.21 ms | 2.34 % |
| Somme des parties | 12117.33 ms | — |
| Temps total boucle | 13178.60 ms | — |
| Écart somme/total | — | 8.05 % |

**Interprétation.** Le vrai goulot d'étranglement n'est pas le calcul GPU (les 2 kernels ne pèsent que 21.7% à eux deux), mais la **boucle CPU qui recalcule `gBest` à chaque itération (65% du temps)**. Le PSO fourni casse une grande partie de l'intérêt du GPU en faisant un aller-retour CPU séquentiel systématique. Conséquence directe pour la conversion en DE : garder **toute la boucle sur le GPU**, avec une seule copie vers le CPU à la fin (cf. flowchart de l'étape 3 du plan).

**Sur l'écart de 8.05% (> 5% demandé).** Il s'explique par le coût propre de l'instrumentation : 8 `cudaEventRecord` + 4 `cudaEventSynchronize` par itération × 30000 itérations = 360 000 appels CUDA ; même ~3µs de surcoût CPU par appel représente ≈1.08s, ce qui correspond quasiment exactement à l'écart observé (1.06s). Ce n'est donc pas un calcul manquant dans l'analyse, mais le coût de la mesure elle-même — limite à mentionner telle quelle dans l'article.

## 1.6 — Questions de compréhension [checkpoint]

Réponds à ces 5 questions (à l'écrit, ici ou dans le chat avec ton assistant) avant de passer à l'étape 2. Elles portent sur `kernel.cu` / `kernel.cpp` / `main.cpp`.

**Q1.** Le calcul du `gBest` final se fait dans une boucle sur CPU (fonction `cuda_pso` dans `kernel.cu`), après chaque appel aux kernels. Pourquoi ce choix plutôt que de calculer `gBest` directement sur GPU ? Quel est l'impact sur la performance (indice : compte le nombre et la taille des `cudaMemcpy` par itération) ?

**Q2.** Le kernel est lancé avec `threadsNum = 32` et `blocksNum = ceil(size / threadsNum)`, où `size = NUM_OF_PARTICLES * NUM_OF_DIMENSIONS`. Avec les valeurs actuelles (512 particules, 3 dimensions), combien de threads sont lancés au total ? À quoi sert le test `if (i >= NUM_OF_PARTICLES * NUM_OF_DIMENSIONS) return;` dans `kernelUpdateParticle` ?

**Q3.** Dans `kernelUpdatePBest`, il y a une condition `i % NUM_OF_DIMENSIONS != 0`. Pourquoi seul un thread par particule (et pas un thread par dimension) doit exécuter la suite de ce kernel ? Que se passerait-il sans cette condition ?

**Q4.** `tempParticle1` et `tempParticle2` sont déclarés `__device__` en mémoire globale du GPU (partagée par tous les threads), pas locaux à un thread. Est-ce correct ici (un seul thread par particule les utilise) ? Que se passerait-il si plusieurs threads par particule y écrivaient en même temps ?

**Q5.** Le protocole final demande de tester `NUM_OF_DIMENSIONS` = 10, 50, 100 et `NUM_OF_PARTICLES` = 50, 100, 500. Au-delà de changer les constantes dans `kernel.h`, quelles autres parties du code doivent changer pour que ça marche correctement (pense à `MAX_ITER`, aux bornes `START_RANGE_MIN/MAX`, et au fait que dim/pop doivent devenir des paramètres passés en ligne de commande) ?

## Réponses (tâche 1.2 — validées avec l'assistant)

**R1.** Le calcul de `gBest` est une réduction (comparer tous les `pBest` entre eux), plus délicate à paralléliser correctement qu'une mise à jour indépendante par particule. Le code choisit la solution simple : rapatrier les `pBest` sur CPU (`cudaMemcpy`), boucler séquentiellement, renvoyer `gBest` au GPU. **Impact** : 2 `cudaMemcpy` supplémentaires à *chaque* itération (sur 30000 itérations), chacun avec un coût de latence fixe non négligeable — mesuré précisément à la tâche 1.3.

**R2.** `size = 512 * 3 = 1536`, `threadsNum = 32` → `blocksNum = ceil(1536/32) = 48`, donc `48*32 = 1536` threads lancés (ici un multiple exact, donc 0 thread hors-limite dans ce cas précis). Le test `if (i >= size) return;` protège contre les cas où `size` n'est pas un multiple de `threadsNum` (ex. si on change `NUM_OF_DIMENSIONS`) : sans lui, des threads du dernier bloc écriraient hors des tableaux → corruption mémoire ou plantage.

**R3.** `kernelUpdatePBest` compare la fitness de la particule *entière* (toutes ses dimensions), pas dimension par dimension. Sans le test `i % NUM_OF_DIMENSIONS != 0`, plusieurs threads de la même particule referaient le même calcul en parallèle et écriraient en même temps dans `tempParticle1`/`tempParticle2` → travail redondant + race condition. Le test garde un seul thread représentant par particule (celui au début de son bloc de dimensions), qui boucle lui-même sur les dimensions.

**R4.** C'est correct *uniquement* grâce à la protection de la Q3 (un seul thread par particule), mais c'est fragile : `tempParticle1`/`tempParticle2` sont des variables `__device__` **globales pour tout le GPU**, pas une par particule. Or plusieurs particules différentes sont traitées en parallèle par des blocs différents en même temps → elles écrivent toutes dans les mêmes variables et s'écrasent mutuellement. C'est un **bug latent** du code fourni : ça semble marcher car l'effet est souvent discret sur le résultat final, mais ce n'est pas thread-safe. En DE, chaque thread devra utiliser une variable locale ou un tableau indexé par thread, jamais une `__device__` globale partagée.

**R5.** Il faut : (1) transformer `NUM_OF_PARTICLES`/`NUM_OF_DIMENSIONS` de `const int` compilés en dur vers des paramètres runtime lus depuis `argc`/`argv` (actuellement lus dans `main.cpp` mais jamais utilisés !), avec allocation dynamique des tableaux (`new`/`malloc` au lieu de tableaux de taille fixe) ; (2) adapter `START_RANGE_MIN`/`MAX` selon la fonction choisie (Rastrigin `[-5,5]`, Rosenbrock `[-100,100]`, Griewank `[-600,600]`, Sphere `[-100,100]`, au lieu du `[-5.12,5.12]` de Levy) ; (3) passer `NUM_OF_PARTICLES`/`NUM_OF_DIMENSIONS` en paramètres aux kernels CUDA (actuellement des constantes globales utilisées en dur dans le code des kernels) puisque la taille des allocations GPU (`cudaMalloc`) en dépend directement. `MAX_ITER = NUM_OF_DIMENSIONS * 10^4` reste une formule valide, il suffit que `NUM_OF_DIMENSIONS` devienne dynamique.